### Exercise 1

In [1]:
import torch

# Construct image with diagonal edges
X = torch.zeros((6, 6))
for i in range(6):
    X[i, i] = 1.0  # diagonal of 1s
X

tensor([[1., 0., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0., 0.],
        [0., 0., 1., 0., 0., 0.],
        [0., 0., 0., 1., 0., 0.],
        [0., 0., 0., 0., 1., 0.],
        [0., 0., 0., 0., 0., 1.]])

In [2]:
def corr2d(X, K):
    h, w = K.shape
    Y = torch.zeros((X.shape[0] - h + 1, X.shape[1] - w + 1))
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            Y[i, j] = (X[i:i+h, j:j+w] * K).sum()
    return Y

K = torch.tensor([[1.0, -1.0]])
corr2d(X, K)

tensor([[ 1.,  0.,  0.,  0.,  0.],
        [-1.,  1.,  0.,  0.,  0.],
        [ 0., -1.,  1.,  0.,  0.],
        [ 0.,  0., -1.,  1.,  0.],
        [ 0.,  0.,  0., -1.,  1.],
        [ 0.,  0.,  0.,  0., -1.]])

In [3]:
corr2d(X.T, K)

tensor([[ 1.,  0.,  0.,  0.,  0.],
        [-1.,  1.,  0.,  0.,  0.],
        [ 0., -1.,  1.,  0.,  0.],
        [ 0.,  0., -1.,  1.,  0.],
        [ 0.,  0.,  0., -1.,  1.],
        [ 0.,  0.,  0.,  0., -1.]])

`X.T` flips the diagonal — but since X is symmetric (the diagonal is the same when transposed), `X.T == X`, so the output is identical to Question 1.

In [4]:
corr2d(X, K.T)

tensor([[ 1., -1.,  0.,  0.,  0.,  0.],
        [ 0.,  1., -1.,  0.,  0.,  0.],
        [ 0.,  0.,  1., -1.,  0.,  0.],
        [ 0.,  0.,  0.,  1., -1.,  0.],
        [ 0.,  0.,  0.,  0.,  1., -1.]])

### Exercise 2

#### Part 1

Yes. The idea is: to detect an edge, you want the kernel to measure the directional derivative of the image intensity in some direction.

If the given vector is

$$
\mathbf{v}=\left(v_1, v_2\right)
$$

then an edge orthogonal to $\mathbf{v}$ means the image changes as you move along $\mathbf{v}$. So the kernel should approximate the derivative in the $\mathbf{v}$-direction.

For an image $I(i, j)$, the directional derivative along v is

$$
v_1 \frac{\partial I}{\partial x}+v_2 \frac{\partial I}{\partial y} .
$$


In discrete form, a simple forward-difference version is:

$$
\frac{\partial I}{\partial x} \approx I(i, j+1)-I(i, j), \quad \frac{\partial I}{\partial y} \approx I(i+1, j)-I(i, j) .
$$


So

$$
v_1 \frac{\partial I}{\partial x}+v_2 \frac{\partial I}{\partial y} \approx v_1(I(i, j+1)-I(i, j))+v_2(I(i+1, j)-I(i, j)) .
$$


Rearranging:

$$
=-\left(v_1+v_2\right) I(i, j)+v_1 I(i, j+1)+v_2 I(i+1, j) .
$$


So one valid edge-detection kernel is

$$
K=\left[\begin{array}{cc}
-\left(v_1+v_2\right) & v_1 \\
v_2 & 0
\end{array}\right] .
$$


This kernel responds strongly when the image changes in direction $\mathbf{v}$, which means it detects edges whose tangent direction is orthogonal to $\mathbf{v}^{\prime}$, namely $\left(v_2,-v_1\right)$.

#### Part 2

Start from the ID second derivative.
For a smooth function $f(x)$, the standard centered finite-difference approximation is

$$
f^{\prime \prime}(x) \approx \frac{f(x+h)-2 f(x)+f(x-h)}{h^2} .
$$


So the corresponding 1D discrete operator is

$$
[1,-2,1]
$$

up to the scaing factor $1 / h^2$.
That already anowers the first part the finite difference operator for the second derivative uses the value at a point, minus twice the center, plus the walue on the other side.

For images, you can apply this in cither horizontal ar vertical drection:

$$
\frac{\partial^2 I}{\partial x^2} \approx \frac{I(i, j+1)-2 I(i, j)+I(i, j-1)}{h^2},
$$

with kernel

$$
\left[\begin{array}{lll}
1 & -2 & 1
\end{array}\right].
$$

and similarly for the vertical direction,

$$
\left[\begin{array}{c}
1 \\
-2 \\
1
\end{array}\right]
$$


If you want the full 2D second-derivative operatar, the most common one is the Laplacian:

$$
\frac{\partial^2 I}{\partial x^2}+\frac{\partial^2 I}{\partial y^2},
$$

which is approximated by

$$
I(i+1, j)+I(i-1, j)+I(i, j+1)+I(i, j-1)-4 I(i, j),
$$

so its kernel is

$$
\left[\begin{array}{ccc}
0 & 1 & 0 \\
1 & -4 & 1 \\
0 & 1 & 0
\end{array}\right]
$$
Minimum kernel size:
- For a second derivative in one drection, the minimum size is 3 entries in 1D.
- For an isotropic 2D second-derivative coperator like the Laplacian, the minimum is $3 \times 3$.

Compared with the first derivative, the second derivative is less about just "there is a change here" and more about "the slope itself is changing sharphy:" That is why it is useful for highlighting fine detail, but also why it is very senaitive to noise.

#### Part 3

A blur kernel should compute a weighted local average of neighboring pixels. Therefore its entries should be nonnegative and sum to 1, so that it smooths the image without greatly changing overall brightness. The simplest example is the $3 \times 3$ box filter

$$
\frac{1}{9}\left[\begin{array}{lll}
1 & 1 & 1 \\
1 & 1 & 1 \\
1 & 1 & 1
\end{array}\right],
$$

which replaces each pixel by the average of itself and its neighbors. A more refined blur kernel is a Gaussian-like kernel such as

$$
\frac{1}{16}\left[\begin{array}{lll}
1 & 2 & 1 \\
2 & 4 & 2 \\
1 & 2 & 1
\end{array}\right],
$$

which gives larger weight to nearby pixels. Such kernels are useful for denoising, suppressing small details, stabilizing edge detection, and anti-aliasing before image downsampling.

#### Part 4

The minimum kernel size for a derivative of order $d$ is $d+1$, because a finite-difference stencil for the $d$-th derivative must cancel all polynomial terms of degree less than $d$, which imposes $d+1$ independent moment conditions and therefore requires at least $d+1$ sample points.

### Exercise 4

The key idea is to **unroll each local patch of the input into a row (or column)**, and **unroll the kernel into a vector**. Then cross-correlation becomes an ordinary matrix-vector multiplication.

#### 1. Start with the 2D cross-correlation formula

Suppose the input is $X \in \mathbb{R}^{n_h \times n_w}$, and the kernel is $K \in \mathbb{R}^{k_h \times k_w}$. Then the output is

$$
Y(i,j)=\sum_{a=0}^{k_h-1}\sum_{b=0}^{k_w-1} X(i+a,j+b),K(a,b).
$$

Each output entry is just the inner product of

* one $k_h \times k_w$ patch of the input, and
* the kernel.

So if we list all patches, we can compute all outputs at once by matrix multiplication.

---

#### 2. Convert the kernel into a vector

Flatten the kernel into a vector:

$$
\mathbf{k} = \operatorname{vec}(K)\in\mathbb{R}^{k_h k_w}.
$$

For example, if

$$
K=
\begin{bmatrix}
k_{11} & k_{12}\\
k_{21} & k_{22}
\end{bmatrix},
$$

then we can write

$$
\mathbf{k}=
\begin{bmatrix}
k_{11}\\
k_{12}\\
k_{21}\\
k_{22}
\end{bmatrix}.
$$

---

#### 3. Convert the input into a patch matrix

Now extract every valid $k_h \times k_w$ window from the input, flatten each one, and stack them as rows of a matrix.

This matrix is often called **im2col**.

If the output size is

$$
(n_h-k_h+1)\times(n_w-k_w+1),
$$

then there are

$$
(n_h-k_h+1)(n_w-k_w+1)
$$

such patches. So define

$$
\mathbf{X}_{\text{col}} \in
\mathbb{R}^{(n_h-k_h+1)(n_w-k_w+1)\times k_h k_w}.
$$

Each row is one flattened local patch.

---

#### 4. Then cross-correlation is a matrix product

Now multiply:

$$
\mathbf{y}=\mathbf{X}_{\text{col}},\mathbf{k}.
$$

This gives a vector containing all output entries. Reshape it into the output shape:

$$
Y = \operatorname{reshape}(\mathbf{y},, n_h-k_h+1,, n_w-k_w+1).
$$

So the full answer is:

$$
\boxed{
\operatorname{vec}(Y)=\mathbf{X}_{\text{col}}\operatorname{vec}(K)
}
$$

up to the convention for row-major or column-major flattening.

---

#### 5. Concrete example

Let

$$
X=
\begin{bmatrix}
0&1&2\\
3&4&5\\
6&7&8
\end{bmatrix},
\qquad
K=
\begin{bmatrix}
0&1\\
2&3
\end{bmatrix}.
$$

The $2\times2$ patches of $X$ are:

top-left:
$$
\begin{bmatrix}
0&1\\
3&4
\end{bmatrix}
$$

top-right:
$$
\begin{bmatrix}
1&2\\
4&5
\end{bmatrix}
$$

bottom-left:
$$
\begin{bmatrix}
3&4\\
6&7
\end{bmatrix}
$$

bottom-right:
$$
\begin{bmatrix}
4&5\\
7&8
\end{bmatrix}
$$

Flatten the kernel:

$$
\mathbf{k}=\left[\begin{array}{l}
0 \\
1 \\
2 \\
3
\end{array}\right] .
$$


Now multiply:

$$
\mathrm{X}_{\mathrm{col}} \mathrm{k}=\left[\begin{array}{cccc}
0 & 1 & 3 & 4 \\
1 & 2 & 4 & 5 \\
3 & 4 & 6 & 7 \\
4 & 5 & 7 & 8
\end{array}\right]\left[\begin{array}{l}
0 \\
1 \\
2 \\
3
\end{array}\right]=\left[\begin{array}{l}
19 \\
25 \\
37 \\
43
\end{array}\right] .
$$

Reshape to $2\times2$:

$$
Y=
\begin{bmatrix}
19&25\\
37&43
\end{bmatrix}.
$$

That matches the cross-correlation output exactly.